# Data Cleaning

This notebook handles data preparation for the weather forecasting project.

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/GlobalWeatherRepository.csv')
print(f'Shape: {df.shape}')
df.head()

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

## Handling Missing Values

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns
categorical_cols = df.select_dtypes(include=['object']).columns

for col in numeric_cols:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)

for col in categorical_cols:
    mode_val = df[col].mode()[0] if len(df[col].mode()) > 0 else 'Unknown'
    df[col] = df[col].fillna(mode_val)

print(f'Missing values after imputation: {df.isnull().sum().sum()}')

## Converting Timestamps

In [ ]:
df['lastupdated'] = pd.to_datetime(df['lastupdated'])
df = df.sort_values('lastupdated').reset_index(drop=True)
print(f'Date range: {df["lastupdated"].min()} to {df["lastupdated"].max()}')

## Removing Outliers (IQR Method)

In [ ]:
def remove_outliers_iqr(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    return df[(df[column] >= lower) & (df[column] <= upper)]

clean_df = df.copy()
for col in ['temperature', 'humidity', 'precipitation']:
    if col in clean_df.columns:
        clean_df = remove_outliers_iqr(clean_df, col)

print(f'Rows after outlier removal: {len(clean_df)}')

In [ ]:
clean_df.to_csv('../data/cleaned_weather.csv', index=False)
print('Saved cleaned data to ../data/cleaned_weather.csv')

## Summary

**What was done:**
- Filled missing numeric values with median (robust to outliers)
- Filled missing categorical values with mode
- Converted lastupdated to datetime and sorted chronologically
- Removed extreme outliers using IQR method

**Why these methods:**
- Median is better than mean for weather data which can have extreme values
- IQR method is simple and effective for removing clear outliers without losing too much data
- Time-based sorting is essential for time series forecasting